In [8]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

### Step 1a - Indexing(Document Ingestion)

In [11]:
video_id = "eMlx5fFNoYc"

try:
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=["en"])

    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In the last chapter, you and I started to step through the internal workings of a transformer. This is one of the key pieces of technology inside large language models, and a lot of other tools in the modern wave of AI. It first hit the scene in a now-famous 2017 paper called Attention is All You Need, and in this chapter you and I will dig into what this attention mechanism is, visualizing how it processes data. As a quick recap, here's the important context I want you to have in mind. The goal of the model that you and I are studying is to take in a piece of text and predict what word comes next. The input text is broken up into little pieces that we call tokens, and these are very often words or pieces of words, but just to make the examples in this video easier for you and me to think about, let's simplify by pretending that tokens are always just words. The first step in a transformer is to associate each token with a high-dimensional vector, what we call its embedding. The most i

### Step 1b - Indexing(Text Splitting)

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

35

### Step 1c and 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [13]:
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

{0: '1d01277a-158b-4cf8-a6f4-9e57a5408c8f',
 1: '26061d40-6a14-4c1d-931a-c2b07f92ca62',
 2: '71acf66d-2108-4d30-892a-122e0845b34a',
 3: 'd2b89d42-6c78-465d-b42b-7ce23e402efb',
 4: '3de0c330-e61a-49f7-bf64-415b52f98d8a',
 5: 'ff5bd880-4f30-4c1b-a6b4-70be5b81266f',
 6: '487c9caf-88bb-4e87-9a63-4de487d6a4fb',
 7: '36a963d8-f61b-4120-9495-995587080b26',
 8: '12439db7-0c0d-4447-aa4f-2c532a6652a7',
 9: '2a58d8dd-a875-4005-9167-399927dfa8fb',
 10: '16ab0b98-35f8-49e4-a825-aa3e99ad977c',
 11: '12ae24b1-88ae-4cdd-9680-19bbc3d1e724',
 12: '31d30822-f50b-491f-a22a-a827fc0602d0',
 13: '2624ccc5-64e0-4cb5-8100-005531fb6855',
 14: 'c5b1233c-d976-4c3a-ad63-dd84ad5bbc40',
 15: '497f11a6-381f-4890-9d22-b75162ac5d01',
 16: 'ff686ccf-8031-4272-b23c-a43826d84b4e',
 17: 'b7587039-7b2b-4433-a503-e5dcf05a1c0a',
 18: '8e8c8e16-0ca7-4667-ba70-7e6fdf699f18',
 19: '03cc5846-b133-42ff-8838-34d23f34a3c2',
 20: '93b983a4-e99d-448e-a799-2f365c4aff3c',
 21: '3b987329-6d18-465a-8861-8dd9c548fa42',
 22: '09d1c28a-9063-

### Step 2 - Retriever

In [14]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [15]:
retriever.invoke("What is Transformer")

[Document(id='1f785af0-aa26-409a-8531-4da4a3b49e25', metadata={}, page_content="about, let's simplify by pretending that tokens are always just words. The first step in a transformer is to associate each token with a high-dimensional vector, what we call its embedding. The most important idea I want you to have in mind is how directions in this high-dimensional space of all possible embeddings can correspond with semantic meaning. In the last chapter we saw an example for how direction can correspond to gender, in the sense that adding a certain step in this space can take you from the embedding of a masculine noun to the embedding of the corresponding feminine noun. That's just one example you could imagine how many other directions in this high-dimensional space could correspond to numerous other aspects of a word's meaning. The aim of a transformer is to progressively adjust these embeddings so that they don't merely encode an individual word, but instead they bake in some much, muc

### Step 3 - Augmentation

In [16]:
llm = GoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.2)

In [17]:
prompt = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.

        {context}
        Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [24]:
question          = "is the topic of llm discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='51998106-7dfd-468e-9836-34046ca8d9b1', metadata={}, page_content="let you do this. If you want to learn more about this stuff, I've left lots of links in the description. In particular, anything produced by Andrej Karpathy or Chris Ola tend to be pure gold. In this video, I wanted to just jump into attention in its current form, but if you're curious about more of the history for how we got here and how you might reinvent this idea for yourself, my friend Vivek just put up a couple videos giving a lot more of that motivation. Also, Britt Cruz from the channel The Art of the Problem has a really nice video about the history of large language models. Thank you."),
 Document(id='ff5bd880-4f30-4c1b-a6b4-70be5b81266f', metadata={}, page_content="saw in the last chapter was how after all of the vectors flow through the network, including many different attention blocks, the computation you perform to produce a prediction of the next token is entirely a function of the last vect

In [25]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [26]:
context_text

"let you do this. If you want to learn more about this stuff, I've left lots of links in the description. In particular, anything produced by Andrej Karpathy or Chris Ola tend to be pure gold. In this video, I wanted to just jump into attention in its current form, but if you're curious about more of the history for how we got here and how you might reinvent this idea for yourself, my friend Vivek just put up a couple videos giving a lot more of that motivation. Also, Britt Cruz from the channel The Art of the Problem has a really nice video about the history of large language models. Thank you.\n\nof the 175 billion that are in the network in total. So even though attention gets all of the attention, the majority of parameters come from the blocks sitting in between these steps. In the next chapter, you and I will talk more about those other blocks and also a lot more about the training process. A big part of the story for the success of the attention mechanism is not so much any spec

In [27]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\n        You are a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n\n        let you do this. If you want to learn more about this stuff, I've left lots of links in the description. In particular, anything produced by Andrej Karpathy or Chris Ola tend to be pure gold. In this video, I wanted to just jump into attention in its current form, but if you're curious about more of the history for how we got here and how you might reinvent this idea for yourself, my friend Vivek just put up a couple videos giving a lot more of that motivation. Also, Britt Cruz from the channel The Art of the Problem has a really nice video about the history of large language models. Thank you.\n\nsaw in the last chapter was how after all of the vectors flow through the network, including many different attention blocks, the computation you perform to produce a prediction of the next token 

### Step 4 - Generation

In [28]:
answer = llm.invoke(final_prompt)
print(answer)

c:\Work\Practice\Generative-Ai-with-LangChain\.venv\lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Yes, large language models (LLMs) are discussed. Specifically, the transcript mentions that Britt Cruz from the channel The Art of the Problem has a really nice video about the history of large language models.
